# TartanIMU — Colab experiments (validated) → final submission

**Runtime → Change runtime type → GPU (A100 if available; an L4 is ~3× slower).** Everything is self-contained: the code cells write the project
files, download the competition data with your Kaggle token, run a grid of *validated* experiments (train on `train`,
score on the disjoint `val` trajectories with the organisers' exact metric, per-platform / per-drone-source breakdown),
and finally refit the best recipe on train+val with a fixed schedule and submit.

Overfitting protocol: every design decision is made on `val` (80 trajectories never seen in training); the final model is
trained on train+val with the *same* number of epochs — no early stopping / checkpoint picking on the leaderboard.

Local reference (Apple M5, 60 epochs): `v4` recipe val **0.2030**, wide `v6` recipe val **0.1987**; public LB of the
160-epoch train+val `v4` model: **0.2870**.

In [ ]:
import os, sys, subprocess, pathlib, json, time
assert subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip(), "No GPU! Runtime -> Change runtime type -> GPU"
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
WORK = pathlib.Path("/content/tartanimu"); WORK.mkdir(exist_ok=True); os.chdir(WORK)
!pip -q install -U kaggle tabulate scipy 2>/dev/null | tail -1
print("workdir:", os.getcwd())

## 1. Kaggle credentials + data
Paste your Kaggle **access token** (`KGAT_...`, from kaggle.com → Settings → API → *Create new token*).
If you only have a `kaggle.json` (username/key), upload it to `/content/tartanimu/kaggle.json` instead and leave the prompt empty.

In [ ]:
from getpass import getpass
tok = getpass("Kaggle access token (KGAT_...), or press Enter if you uploaded kaggle.json: ").strip()
kdir = pathlib.Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
if tok:
    (kdir / "access_token").write_text(tok); os.chmod(kdir / "access_token", 0o600)
elif pathlib.Path("kaggle.json").exists():
    import shutil; shutil.copy("kaggle.json", kdir / "kaggle.json"); os.chmod(kdir / "kaggle.json", 0o600)
else:
    raise SystemExit("no credentials")
!kaggle competitions list -s tartan | head -3

In [ ]:
if not pathlib.Path("data/index/train_windows.csv").exists():
    !kaggle competitions download -c tartan-imu-challenge-iros2026 -p data -q
    !cd data && unzip -q -o tartan-imu-challenge-iros2026.zip && rm tartan-imu-challenge-iros2026.zip
!ls data data/index | head; du -sh data

## 2. Project code (identical to the repository)

In [ ]:
%%writefile common.py
"""Shared data utilities for the TartanIMU challenge.

Data conventions (from the official starter kit):
  imu (N,6) = [ax, ay, az, gx, gy, gz]  (m/s^2, rad/s), 200 Hz, body frame, gravity retained
  window k of a trajectory = imu[k*200:(k+1)*200]; target = mean vel_body over that window
  platform_id: car=0, dog=1, drone=2, human=3
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent
DATA = ROOT / "data"
WIN = 200
PLATFORMS = ["car", "dog", "drone", "human"]
PLAT2ID = {p: i for i, p in enumerate(PLATFORMS)}


def read_index(split: str) -> pd.DataFrame:
    """window index (+ targets for train/val), one row per window, sorted by traj/win_idx."""
    df = pd.read_csv(DATA / "index" / f"{split}_windows.csv").dropna(axis=1, how="all")
    if split != "test":
        df = df.merge(pd.read_csv(DATA / "index" / f"{split}_targets.csv"), on="window_id")
    return df.sort_values(["traj_id", "win_idx"]).reset_index(drop=True)


def traj_path(split: str, traj_id: str) -> Path:
    if split == "test":
        return DATA / "test" / f"{traj_id}.npz"
    return DATA / split / traj_id.split("_")[0] / f"{traj_id}.npz"


def load_split(split: str, keys=("imu",)) -> dict[str, dict[str, np.ndarray]]:
    """Load every trajectory of a split into memory: {traj_id: {key: array}}."""
    idx = read_index(split)
    out = {}
    for tid in idx["traj_id"].unique():
        with np.load(traj_path(split, tid)) as d:
            out[tid] = {k: d[k] for k in keys if k in d.files}
            out[tid]["n_win"] = len(d["ts"]) // WIN
    return out


def build_solution(split: str) -> pd.DataFrame:
    """Build the scorer's `solution` frame for a labelled split (val/train).

    Per window: quaternion at the window's mid sample, ground-truth position at the window's
    last sample (the integrated path point), dt = window duration.
    """
    idx = read_index(split)
    rows = []
    for tid, g in idx.groupby("traj_id", sort=False):
        with np.load(traj_path(split, tid)) as d:
            quat, pos, ts, fs = d["quat"], d["pos"], d["ts"], float(d["fs"])
        w = g["win_idx"].to_numpy()
        s, e, m = w * WIN, w * WIN + WIN - 1, w * WIN + WIN // 2
        rows.append(pd.DataFrame({
            "window_id": g["window_id"].to_numpy(), "traj_id": tid, "win_idx": w,
            "platform": g["platform"].to_numpy(),
            "qx": quat[m, 0], "qy": quat[m, 1], "qz": quat[m, 2], "qw": quat[m, 3],
            "gx": pos[e, 0], "gy": pos[e, 1], "gz": pos[e, 2],
            "dt": ts[e] - ts[s] + 1.0 / fs,
            "vx_gt": g["vx"].to_numpy(), "vy_gt": g["vy"].to_numpy(), "vz_gt": g["vz"].to_numpy(),
        }))
    return pd.concat(rows, ignore_index=True)


def score_predictions(solution: pd.DataFrame, pred: pd.DataFrame, per_platform: bool = True):
    """Official TartanIMU score + per-platform AVE / ATE20 breakdown."""
    from kaggle_metric import score, _ate_traj, _ave_traj, AVE_REF, ATE_REF

    total = score(solution, pred, "window_id")
    if not per_platform:
        return total, None
    m = solution.merge(pred.rename(columns={"vx": "vx_pred", "vy": "vy_pred", "vz": "vz_pred"}), on="window_id")
    recs = []
    for tid, g in m.groupby("traj_id", sort=False):
        g = g.sort_values("win_idx")
        recs.append({"platform": g["platform"].iloc[0], "traj_id": tid, "ate20": _ate_traj(g), "ave": _ave_traj(g)})
    pp = pd.DataFrame(recs).groupby("platform")[["ave", "ate20"]].mean()
    pp["score"] = 0.6 * pp["ave"] / AVE_REF + 0.4 * pp["ate20"] / ATE_REF
    return total, pp


In [ ]:
%%writefile kaggle_metric.py
"""TartanIMU / IMUNet Challenge — Kaggle custom scoring metric: **TartanIMU Score**.

    TartanIMU Score = 0.6 * (macro AVE / AVE_REF)  +  0.4 * (macro ATE20 / ATE_REF)

Lower is better. The score is dimensionless: each component is divided by the value that
the all-zero submission attains on the full test set, so the two terms — one in m/s, one
in metres — are put on a common scale before they are combined. Two consequences worth
stating plainly:

* **The all-zero submission scores exactly 1.000.** Anything above 1.0 is worse than
  submitting nothing at all.
* **The declared 0.6 / 0.4 weights are the weights that actually act.** Without
  normalisation a raw `0.6*AVE + 0.4*ATE` would be dominated by ATE, because ATE spans
  roughly 0.15 - 3.5 m while AVE spans only 0.0 - 0.75 m/s; the nominal 60 % term would
  carry about 25 % of the discriminative power. After normalisation the two terms
  contribute 59 % / 41 % of the spread across our reference submissions, matching intent.

The two components measure genuinely different failure modes, which is why both are kept:

* **AVE — Absolute Velocity Error** (m/s). The mean, over a trajectory's windows, of the
  Euclidean norm ||v_pred - v_gt||. This is instantaneous, per-window accuracy: it
  rewards getting the speed and the direction of motion right at each moment, and it is
  the quantity a downstream state estimator actually consumes.
* **ATE20 — 20 m-segment Absolute Trajectory Error** (m). The predicted body-frame
  velocities are integrated into a path and compared with the ground-truth path over
  fixed 20 m pieces after an SE(3) alignment. This is temporal consistency: it punishes
  correlated bias and drift that a per-window average error hides.

A model can be good at one and bad at the other. Low AVE with high ATE means small but
systematically signed errors that accumulate; low ATE with high AVE means noisy
predictions whose errors happen to cancel. Requiring both is what makes the benchmark
about deployable inertial odometry rather than about either statistic alone.

Both components use the same hierarchical averaging: per window -> mean over a
trajectory's windows/segments -> mean over a platform's trajectories -> equal-weight mean
over the four platforms (macro-average), so no platform dominates and each contributes
exactly 25 % of each component.

ATE20 details. Per test trajectory the ground-truth path is cut into consecutive segments
of ~20 m of travelled distance. Within each segment the participant's per-window
body-frame velocities are rotated to the world frame with ground-truth orientation (used
ONLY in this scoring step, never as a model input, because global heading is not
observable from an IMU alone), multiplied by the window duration, accumulated into a path,
SE(3)-aligned to the ground-truth path over that segment (Umeyama, rotation + translation,
no scale), and the RMS position error is the segment's ATE.

Why segments rather than whole trajectories: a whole-trajectory ATE rewards predicting
nothing. An all-zero submission integrates to a single point, whose aligned error is just
the ground-truth path's radius of gyration — on spatially compact trajectories (a person
pacing indoors, a drone circling) that beats an honest model which accumulates drift.
Fixing the segment length to 20 m of travel removes that degenerate strategy: every scored
segment covers the same ground, so standing still is never cheap. The length is 20 m
rather than 5 m to keep the alignment well conditioned on every platform: the platforms
differ by an order of magnitude in speed, and at 5 m the fastest one is down to ~4
one-second windows per segment, where a nearly straight flight leaves the Umeyama rotation
poorly constrained.

Reference scores (organizers' own submissions, full test set):

    submission                       ATE20     AVE   TartanIMU Score
    ground-truth velocities (floor)  0.151   0.000            0.019
    organizers' unified baseline     1.261   0.461            0.538
    all-zeros (sample_submission)    3.116   0.736            1.000
    per-platform mean velocity       3.496   0.749            1.060

The metric is shrink-proof: scaling the baseline's predictions by any factor away from 1.0
strictly worsens the score (verified over 0.0 - 1.5), so submitting deliberately damped
velocities is never advantageous.

Submission columns: window_id, vx, vy, vz            (one body-frame velocity per window)
Solution columns  : window_id, traj_id, win_idx, platform, qx, qy, qz, qw,
                    gx, gy, gz, dt, vx_gt, vy_gt, vz_gt, Usage
"""
import numpy as np
import pandas as pd

SEGMENT_LENGTH_M = 20.0
MIN_SEGMENT_POINTS = 3          # Umeyama needs three points to pin down a rotation

W_AVE = 0.6                     # weight of the velocity-accuracy term
W_ATE = 0.4                     # weight of the trajectory-consistency term

# Value each component attains for the all-zero submission on the full test set. Fixed,
# published constants — they put m/s and m on a common scale and pin the all-zero
# submission to exactly 1.000. They are properties of the test set, not tunables.
AVE_REF = 0.7356384388          # m/s
ATE_REF = 3.1160277267          # m


class ParticipantVisibleError(Exception):
    """Message shown to the participant on the leaderboard."""


def _quat_to_R(q: np.ndarray) -> np.ndarray:
    """(M,4) quaternion (x, y, z, w) -> (M,3,3) rotation matrices."""
    x, y, z, w = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
    n = np.sqrt(x * x + y * y + z * z + w * w)
    n[n == 0] = 1.0
    x, y, z, w = x / n, y / n, z / n, w / n
    R = np.empty((q.shape[0], 3, 3))
    R[:, 0, 0] = 1 - 2 * (y * y + z * z); R[:, 0, 1] = 2 * (x * y - z * w); R[:, 0, 2] = 2 * (x * z + y * w)
    R[:, 1, 0] = 2 * (x * y + z * w); R[:, 1, 1] = 1 - 2 * (x * x + z * z); R[:, 1, 2] = 2 * (y * z - x * w)
    R[:, 2, 0] = 2 * (x * z - y * w); R[:, 2, 1] = 2 * (y * z + x * w); R[:, 2, 2] = 1 - 2 * (x * x + y * y)
    return R


def _umeyama_align(P: np.ndarray, Q: np.ndarray) -> tuple:
    """Least-squares SE(3) (R, t), no scale, aligning P onto Q. P,Q are (M,3)."""
    muP, muQ = P.mean(0), Q.mean(0)
    Pc, Qc = P - muP, Q - muQ
    H = Pc.T @ Qc
    U, _, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1.0, 1.0, d])
    R = Vt.T @ D @ U.T
    t = muQ - R @ muP
    return R, t


def _segment_bounds(Q: np.ndarray, seg_len: float = SEGMENT_LENGTH_M,
                    min_pts: int = MIN_SEGMENT_POINTS) -> list:
    """Cut a ground-truth path into ~seg_len-metre pieces.

    Returns (start, end) index pairs, end inclusive. Boundaries fall where the cumulative
    travelled distance crosses another seg_len; the tail is kept as its own segment. Any
    piece with fewer than min_pts points is merged into its neighbour, so every returned
    segment can support an SE(3) alignment. A trajectory shorter than seg_len yields one
    segment covering all of it.
    """
    cum = np.concatenate([[0.0], np.cumsum(np.linalg.norm(np.diff(Q, axis=0), axis=1))])
    cuts, reached = [], 0.0
    for i, c in enumerate(cum):
        if c >= reached + seg_len:
            cuts.append(i)
            reached = c
    if not cuts or cuts[-1] != len(Q) - 1:
        cuts.append(len(Q) - 1)

    segs, start = [], 0
    for end in cuts:
        if end > start:
            segs.append([start, end])
            start = end
    if not segs:
        return []

    merged = [segs[0]]
    for s, e in segs[1:]:
        if e - s + 1 < min_pts:                       # short tail -> extend the previous
            merged[-1][1] = e
        else:
            merged.append([s, e])
    if merged[0][1] - merged[0][0] + 1 < min_pts and len(merged) > 1:
        merged[1][0] = merged[0][0]                   # short head -> fold into the next
        merged.pop(0)
    return [(s, e) for s, e in merged if e - s + 1 >= min_pts]


def _ate_segment(R: np.ndarray, v: np.ndarray, dt: np.ndarray, Q: np.ndarray) -> float:
    """RMS position error of one aligned segment."""
    disp = np.einsum("mij,mj->mi", R, v) * dt
    P = np.cumsum(disp, axis=0)
    Rr, t = _umeyama_align(P, Q)
    return float(np.sqrt(np.mean(np.sum((P @ Rr.T + t - Q) ** 2, axis=1))))


def _ate_traj(g: pd.DataFrame) -> float:
    """Mean 20 m-segment ATE for one trajectory dataframe (already time-ordered)."""
    Q = g[["gx", "gy", "gz"]].to_numpy(float)
    bounds = _segment_bounds(Q)
    if not bounds:
        return float("nan")
    R = _quat_to_R(g[["qx", "qy", "qz", "qw"]].to_numpy(float))
    v = g[["vx_pred", "vy_pred", "vz_pred"]].to_numpy(float)
    dt = g["dt"].to_numpy(float)[:, None]
    return float(np.mean([_ate_segment(R[s:e + 1], v[s:e + 1], dt[s:e + 1], Q[s:e + 1])
                          for s, e in bounds]))


def _ave_traj(g: pd.DataFrame) -> float:
    """Mean per-window Euclidean velocity error (m/s) for one trajectory."""
    err = (g[["vx_pred", "vy_pred", "vz_pred"]].to_numpy(float)
           - g[["vx_gt", "vy_gt", "vz_gt"]].to_numpy(float))
    return float(np.mean(np.linalg.norm(err, axis=1)))


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """TartanIMU Score = 0.6*(macro AVE / AVE_REF) + 0.4*(macro ATE20 / ATE_REF).

    Both components are macro-averaged over the four platforms, so each platform carries
    exactly 25 % of each term. AVE is the mean per-window Euclidean velocity error; ATE20
    integrates the predicted body-frame velocities over fixed 20 m pieces of the
    ground-truth path and takes the RMS position error after an SE(3) alignment. Each is
    divided by the value the all-zero submission attains on the full test set, which makes
    the score dimensionless and pins the all-zero submission to 1.000.

    Args:
        solution: window_id, traj_id, win_idx, platform, qx, qy, qz, qw, gx, gy, gz, dt,
            vx_gt, vy_gt, vz_gt (Usage already removed by Kaggle).
        submission: window_id, vx, vy, vz.
        row_id_column_name: name of the id column ("window_id").

    Returns:
        The TartanIMU Score as a finite float; lower is better.
    """
    for col in (row_id_column_name, "vx", "vy", "vz"):
        if col not in submission.columns:
            raise ParticipantVisibleError(f"Submission must contain column '{col}'.")

    sub = submission[[row_id_column_name, "vx", "vy", "vz"]].copy()
    for c in ("vx", "vy", "vz"):
        sub[c] = pd.to_numeric(sub[c], errors="coerce")
    if sub[["vx", "vy", "vz"]].isnull().any().any():
        raise ParticipantVisibleError("Submission contains non-numeric or missing vx/vy/vz values.")
    if not np.isfinite(sub[["vx", "vy", "vz"]].to_numpy(float)).all():
        raise ParticipantVisibleError("Submission contains infinite vx/vy/vz values.")
    if sub[row_id_column_name].duplicated().any():
        raise ParticipantVisibleError(f"Submission contains duplicate '{row_id_column_name}' values.")

    need = {"traj_id", "win_idx", "platform", "qx", "qy", "qz", "qw",
            "gx", "gy", "gz", "dt", "vx_gt", "vy_gt", "vz_gt"}
    if not need.issubset(solution.columns):
        raise ParticipantVisibleError("Solution file is missing required scoring columns.")

    m = solution.merge(sub.rename(columns={"vx": "vx_pred", "vy": "vy_pred", "vz": "vz_pred"}),
                       on=row_id_column_name, how="left")
    if m[["vx_pred", "vy_pred", "vz_pred"]].isnull().any().any():
        raise ParticipantVisibleError("Submission is missing one or more required window_id rows.")

    per_platform = {}
    for _, g in m.groupby("traj_id", sort=False):
        g = g.sort_values("win_idx")
        ate, ave = _ate_traj(g), _ave_traj(g)
        if np.isfinite(ate) and np.isfinite(ave):
            per_platform.setdefault(g["platform"].iloc[0], []).append((ate, ave))

    if not per_platform:
        raise ParticipantVisibleError("No scorable trajectories found.")

    macro_ate = float(np.mean([np.mean([x[0] for x in v]) for v in per_platform.values()]))
    macro_ave = float(np.mean([np.mean([x[1] for x in v]) for v in per_platform.values()]))

    result = W_AVE * (macro_ave / AVE_REF) + W_ATE * (macro_ate / ATE_REF)
    if not np.isfinite(result):
        raise ParticipantVisibleError("Score is not finite; check for extreme velocity values.")
    return result


In [ ]:
%%writefile model.py
"""Unified context model for multi-platform inertial odometry.

Input : raw 200 Hz IMU chunk (B, 6, L) covering T = L/200 consecutive windows of one trajectory.
Output: dense body-frame velocity at 20 Hz (B, T*20, 3) -> averaged to one velocity per 1 s window,
        plus platform logits (auxiliary supervision only; never used for routing).

Architecture: strided conv stem (200 Hz -> 20 Hz tokens) -> dilated depthwise TCN (~13 s receptive
field) -> 2-layer transformer encoder (whole-chunk context, past and future) -> per-token velocity head.
One shared set of weights handles all four embodiments.
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from common import WIN

TOK = 20                                  # tokens per 1 s window (200 Hz / 10)
CHUNK_WIN = 16                            # windows per training / inference chunk
# fixed input scaling (raw units -> O(1)); zero mean keeps rotation augmentation exact
IN_SCALE = torch.tensor([3.0, 3.0, 3.0, 0.4, 0.4, 0.4]).view(1, 6, 1)


class TCNBlock(nn.Module):
    def __init__(self, w: int, dilation: int, drop: float = 0.1):
        super().__init__()
        self.norm = nn.GroupNorm(8, w)
        self.dw = nn.Conv1d(w, w, 5, padding=2 * dilation, dilation=dilation, groups=w, bias=False)
        self.pw = nn.Sequential(nn.Conv1d(w, 3 * w, 1), nn.GELU(), nn.Dropout(drop), nn.Conv1d(3 * w, w, 1))

    def forward(self, x):
        return x + self.pw(self.dw(self.norm(x)))


def _exp_so3(w):
    """Batched Rodrigues: rotation vectors (..., 3) -> matrices (..., 3, 3)."""
    th = w.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    k = w / th
    K = torch.zeros(*w.shape[:-1], 3, 3, device=w.device, dtype=w.dtype)
    K[..., 0, 1], K[..., 0, 2], K[..., 1, 0] = -k[..., 2], k[..., 1], k[..., 2]
    K[..., 1, 2], K[..., 2, 0], K[..., 2, 1] = -k[..., 0], -k[..., 1], k[..., 0]
    I = torch.eye(3, device=w.device, dtype=w.dtype).expand_as(K)
    st, ct = th.sin()[..., None], th.cos()[..., None]
    return I + st * K + (1 - ct) * (K @ K)


@torch.no_grad()
def physics_features(x, dt_tok: float = TOK / WIN / TOK * 10):
    """Deterministic strap-down features at token rate from a raw chunk x (B, 6, L).

    Gyro is integrated (parallel prefix product of per-token rotations) to get R_t: body(t) -> body(chunk start).
    Accelerometer is rotated into that common frame, its chunk mean is taken as the gravity estimate, and the
    de-gravitated acceleration is integrated to a relative velocity.  Everything is then expressed back in the
    *current* body frame, so all outputs are ordinary body-frame vectors (rotation-equivariant like the target):
      a_dyn (3): accelerometer minus estimated gravity,   g_b (3): estimated gravity direction in body frame,
      dv (3):    velocity change since chunk start.       Returns (B, 9, N) with N = L/10 tokens.
    """
    B, _, L = x.shape
    N = L // 10
    xb = x.float().view(B, 6, N, 10).mean(-1)                       # 20 Hz
    acc, gyr = xb[:, :3].transpose(1, 2), xb[:, 3:].transpose(1, 2)  # (B, N, 3)
    dt = 10.0 / 200.0
    E = _exp_so3(gyr * dt)                                          # per-token increments
    P = E.clone(); d = 1                                            # Hillis-Steele scan: P[t] = E_1 ... E_t
    while d < N:
        P = torch.cat([P[:, :d], P[:, :-d] @ P[:, d:]], dim=1); d *= 2
    R = P                                                           # (B, N, 3, 3): body(t) -> body(0)
    a0 = (R @ acc[..., None])[..., 0]                               # accel in start frame
    g0 = a0.mean(1, keepdim=True)                                   # gravity estimate (start frame)
    v0 = torch.cumsum(a0 - g0, dim=1) * dt                          # relative velocity (start frame)
    Rt = R.transpose(-1, -2)
    dv = (Rt @ v0[..., None])[..., 0]
    gb = (Rt @ g0.expand_as(a0)[..., None])[..., 0]
    a_dyn = acc - gb
    return torch.cat([a_dyn / 3.0, gb / 9.81, dv / 3.0], dim=-1).transpose(1, 2)  # (B, 9, N)


class IMUNet(nn.Module):
    def __init__(self, width: int = 128, blocks: int = 8, ctx_layers: int = 2, drop: float = 0.1, max_tok: int = 4096,
                 physics: bool = False, lag: bool = False):
        super().__init__()
        self.physics, self.lag = physics, lag
        self.register_buffer("in_scale", IN_SCALE.clone())
        if physics:
            self.phys_proj = nn.Sequential(nn.Conv1d(18, width, 1), nn.GELU(), nn.Conv1d(width, width, 1))
        self.stem = nn.Sequential(
            nn.Conv1d(6, width // 2, 9, stride=2, padding=4, bias=False), nn.GroupNorm(8, width // 2), nn.GELU(),
            nn.Conv1d(width // 2, width, 11, stride=5, padding=5, bias=False), nn.GroupNorm(8, width), nn.GELU())
        dil = (1, 2, 4, 8, 16, 32, 1, 2)
        self.tcn = nn.Sequential(*[TCNBlock(width, dil[i % len(dil)], drop) for i in range(blocks)])
        self.pos = nn.Parameter(torch.zeros(1, max_tok, width)); nn.init.trunc_normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(width, 4, 3 * width, dropout=drop, activation="gelu", batch_first=True, norm_first=True)
        self.ctx = nn.TransformerEncoder(layer, ctx_layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 128), nn.GELU(), nn.Linear(128, 3))
        self.attn = nn.Linear(width, 1)
        self.plat = nn.Sequential(nn.LayerNorm(2 * width), nn.Linear(2 * width, 64), nn.GELU(), nn.Linear(64, 4))
        if lag:   # per-chunk IMU-vs-ground-truth time offset (in 50 ms tokens); some recordings are offset by up to 70 ms
            self.lag_head = nn.Sequential(nn.LayerNorm(2 * width), nn.Linear(2 * width, 64), nn.GELU(), nn.Linear(64, 1))
            nn.init.zeros_(self.lag_head[-1].weight); nn.init.zeros_(self.lag_head[-1].bias)

    @staticmethod
    def shift_dense(dense, delta):
        """Fractional time shift of a (B, N, 3) sequence: out[t] = dense[t + delta], delta (B,) in tokens, edge-clamped."""
        B, N, _ = dense.shape
        pos = torch.arange(N, device=dense.device, dtype=dense.dtype)[None] + delta[:, None]      # (B, N)
        pos = pos.clamp(0, N - 1)
        i0 = pos.floor().long(); i1 = (i0 + 1).clamp(max=N - 1); w = (pos - i0.to(dense.dtype))[..., None]
        g0 = torch.gather(dense, 1, i0[..., None].expand(B, N, 3)); g1 = torch.gather(dense, 1, i1[..., None].expand(B, N, 3))
        return g0 * (1 - w) + g1 * w

    def forward(self, x, return_lag: bool = False):                         # x: (B, 6, L) raw IMU
        h = self.stem(x / self.in_scale)            # (B, W, L/10)
        if self.physics:
            # two hypotheses for the gyro-z sign (one drone source has it inverted); the network learns which to trust
            xz = torch.cat([x[:, :5], -x[:, 5:6]], dim=1)
            h = h + self.phys_proj(torch.cat([physics_features(x), physics_features(xz)], dim=1))
        h = self.tcn(h)
        h = h.transpose(1, 2)                       # (B, N, W)
        h = self.ctx(h + self.pos[:, : h.shape[1]])
        dense = self.head(h)                        # (B, N, 3)  20 Hz velocity
        a = torch.softmax(self.attn(h), dim=1)
        desc = torch.cat([(h * a).sum(1), h.mean(1)], dim=-1)
        plat = self.plat(desc)
        delta = None
        if self.lag:
            delta = 2.0 * torch.tanh(self.lag_head(desc)[:, 0])                # +-2 tokens = +-100 ms
            dense = self.shift_dense(dense, delta)
        if return_lag:
            return dense, plat, delta
        return dense, plat

    @staticmethod
    def to_windows(dense):                          # (B, N, 3) -> (B, N/TOK, 3)
        B, N, _ = dense.shape
        return dense.view(B, N // TOK, TOK, 3).mean(2)


# --------------------------------------------------------------------------- trajectory inference
@torch.no_grad()
def predict_trajectory(model: nn.Module, imu: np.ndarray, device, chunk_win: int = CHUNK_WIN, stride_win: int = 4,
                       batch: int = 32) -> np.ndarray:
    """Sliding-chunk inference over one whole trajectory. Returns (n_win, 3) window velocities.

    Overlapping chunks are averaged with a raised-cosine weight so each window is trusted most from the
    chunk where it sits near the centre (full bidirectional context). Short trajectories are edge-padded.
    """
    n_win = len(imu) // WIN
    x = imu[: n_win * WIN].astype(np.float32)
    pad = max(0, chunk_win - n_win)
    if pad:
        x = np.concatenate([x, np.repeat(x[-1:], pad * WIN, axis=0)])
    total = n_win + pad
    starts = list(range(0, total - chunk_win + 1, stride_win))
    if starts[-1] != total - chunk_win:
        starts.append(total - chunk_win)
    w = 0.5 - 0.5 * np.cos(2 * np.pi * (np.arange(chunk_win) + 0.5) / chunk_win)   # Hann, >0 everywhere
    w = w.astype(np.float32) + 0.05
    acc = np.zeros((total, 3), np.float32)
    wsum = np.zeros((total, 1), np.float32)
    for i in range(0, len(starts), batch):
        ss = starts[i:i + batch]
        xb = np.stack([x[s * WIN:(s + chunk_win) * WIN].T for s in ss])
        dense, _ = model(torch.from_numpy(xb).to(device))
        vb = model.to_windows(dense).float().cpu().numpy()
        for s, v in zip(ss, vb):
            acc[s:s + chunk_win] += v * w[:, None]
            wsum[s:s + chunk_win] += w[:, None]
    return (acc / wsum)[:n_win]


In [ ]:
%%writefile train.py
"""Train the unified IMU velocity model and validate with the official TartanIMU scorer.

  python train.py                          # train on train, validate on val (model selection)
  python train.py --splits train,val       # final fit on everything (fixed schedule, no selection)

Checkpoints + logs go to ./runs/<name>/.
"""
from __future__ import annotations

import argparse, json, math, time
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from common import WIN, PLATFORMS, PLAT2ID, read_index, load_split, build_solution, score_predictions
from model import IMUNet, CHUNK_WIN, TOK, predict_trajectory

p = argparse.ArgumentParser()
p.add_argument("--name", default="ctx_tcn_gru")
p.add_argument("--splits", default="train")
p.add_argument("--epochs", type=int, default=40)
p.add_argument("--steps", type=int, default=300, help="optimizer steps per epoch")
p.add_argument("--batch", type=int, default=64)
p.add_argument("--lr", type=float, default=1.5e-3)
p.add_argument("--wd", type=float, default=0.02)
p.add_argument("--chunk", type=int, default=CHUNK_WIN)
p.add_argument("--width", type=int, default=128)
p.add_argument("--ctx-layers", type=int, default=2)
p.add_argument("--seed", type=int, default=0)
p.add_argument("--eval-every", type=int, default=2)
p.add_argument("--rot-deg", type=float, default=15.0, help="max sensor-mount rotation augmentation")
p.add_argument("--physics", type=int, default=0, help="1 = add gyro-integrated strap-down features")
p.add_argument("--dilate", type=float, default=1.0, help="time-dilation aug: speed factor ~ logU(1/x, x); 1 = off")
p.add_argument("--traj-uniform", type=int, default=0, help="1 = sample trajectories uniformly within a platform (metric weights trajectories equally), instead of by length")
p.add_argument("--drop-source-b", type=int, default=0, help="1 = exclude the second drone source (train idx > 42 / val idx > 8) from training")
p.add_argument("--lag", type=int, default=0, help="1 = learned per-chunk IMU/GT time-shift head, supervised by measured lags")
p.add_argument("--vib", type=float, default=1.0, help="vibration aug: scale the >~20 Hz part of the IMU by logU(1/x, x) per chunk (1 = off)")
p.add_argument("--rot-deg-drone", type=float, default=-1, help="max rotation aug for drone chunks (-1 = same as --rot-deg)")
p.add_argument("--boost-a", type=float, default=1.0, help="sampling weight multiplier for source-A drone trajectories (index <= 42 train / <= 8 val)")
p.add_argument("--swa-from", type=int, default=0, help="if >0, average EMA weights over epochs >= this into runs/<name>/swa.pt")
p.add_argument("--plat-probs", default="0.25,0.25,0.25,0.25", help="sampling probability per platform (car,dog,drone,human)")
args = p.parse_args()

torch.manual_seed(args.seed); np.random.seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
run = Path("runs") / args.name; run.mkdir(parents=True, exist_ok=True)
print("device", device, "| run", run, "| args", vars(args))

# --------------------------------------------------------------------------- data
T = args.chunk
trajs = []                                                # list of dicts: imu, win_v, dense_v, plat, n_win
from scipy.spatial.transform import Rotation


def measure_lag_tokens(imu, quat, fs=200, max_lag=30):
    """IMU-vs-GT time offset from |gyro| vs |GT angular rate| cross-correlation; returns tokens (50 ms), sign so that
    GT(t) ~ f(IMU(t + lag)).  Negative = the IMU leads the ground truth."""
    n = min(len(quat), 24000); Rq = Rotation.from_quat(quat[:n])
    gb = np.linalg.norm((Rq[:-1].inv() * Rq[1:]).as_rotvec() * fs, axis=1); ga = np.linalg.norm(imu[:n - 1, 3:], axis=1)
    ga, gb = ga - ga.mean(), gb - gb.mean(); best, best_l = -2, 0
    for l in range(-max_lag, max_lag + 1):
        x, y = (ga[l:], gb[:len(gb) - l]) if l >= 0 else (ga[:l], gb[-l:])
        c = (x * y).sum() / np.sqrt((x * x).sum() * (y * y).sum() + 1e-12)
        if c > best: best, best_l = c, l
    return float(np.clip(best_l / 10.0, -2, 2))


for split in args.splits.split(","):
    for tid, d in load_split(split, keys=("imu", "vel_body", "quat")).items():
        if args.drop_source_b and tid.startswith("drone") and int(tid[-4:]) > (42 if split == "train" else 8):
            continue
        n = d["n_win"]
        vb = d["vel_body"][: n * WIN]
        rec = {"id": tid, "imu": d["imu"][: n * WIN].astype(np.float32), "n_win": n,
               "win_v": vb.reshape(n, WIN, 3).mean(1).astype(np.float32),
               "dense_v": vb.reshape(n * TOK, WIN // TOK, 3).mean(1).astype(np.float32),
               "plat": PLAT2ID[tid.split("_")[0]]}
        rec["lag"] = measure_lag_tokens(d["imu"], d["quat"]) if args.lag else 0.0
        if args.dilate > 1:                                   # gravity in the body frame, needed to dilate physically
            rec["vel"] = vb.astype(np.float32)
            rec["grav"] = Rotation.from_quat(d["quat"][: n * WIN]).inv().apply(np.tile([0.0, 0.0, 9.81], (n * WIN, 1))).astype(np.float32)
        trajs.append(rec)
def _is_source_a(tid):                                       # racing-drone source: fast, multi-mount, tiny
    return tid.startswith("drone") and int(tid[-4:]) <= (42 if "train" in tid else 8)
by_plat = {i: [k for k, t in enumerate(trajs) if t["plat"] == i] for i in range(4)}
plat_w = {i: np.array([(1.0 if args.traj_uniform else trajs[k]["n_win"]) * (args.boost_a if _is_source_a(trajs[k]["id"]) else 1.0) for k in ks], float) for i, ks in by_plat.items()}
plat_w = {i: w / w.sum() for i, w in plat_w.items()}
print({PLATFORMS[i]: len(ks) for i, ks in by_plat.items()}, "trajectories;", sum(t["n_win"] for t in trajs), "windows")
if args.lag:
    print("measured lag (tokens) by platform:", {PLATFORMS[i]: np.round(np.mean([trajs[k]["lag"] for k in ks]), 2) for i, ks in by_plat.items()})

rng = np.random.default_rng(args.seed)
plat_probs = np.array([float(x) for x in args.plat_probs.split(",")]); plat_probs /= plat_probs.sum()


def sample_batch(B):
    """Platform-balanced random chunks of T windows (edge-padded + masked if trajectory is shorter)."""
    X = np.empty((B, T * WIN, 6), np.float32); Yw = np.empty((B, T, 3), np.float32)
    Yd = np.empty((B, T * TOK, 3), np.float32); M = np.ones((B, T), np.float32); P = np.empty(B, np.int64); LG = np.zeros(B, np.float32)
    for b in range(B):
        pl = rng.choice(4, p=plat_probs)
        t = trajs[rng.choice(by_plat[pl], p=plat_w[pl])]
        n = t["n_win"]
        L = T * WIN
        if args.dilate > 1 and n * WIN >= int(L * args.dilate) + 1:
            # physical time dilation by speed factor sp: v -> sp v, gyro -> sp w, dynamic accel -> sp^2 (f - g)
            sp = float(np.exp(rng.uniform(-np.log(args.dilate), np.log(args.dilate))))
            Lr = int(round(L * sp))
            s = rng.integers(0, n * WIN - Lr + 1)
            src = np.arange(Lr, dtype=np.float32); q = np.linspace(0, Lr - 1, L, dtype=np.float32)
            imu, vel, grav = (np.stack([np.interp(q, src, a[s:s + Lr, c]) for c in range(a.shape[1])], 1) for a in (t["imu"], t["vel"], t["grav"]))
            X[b, :, :3] = grav + sp * sp * (imu[:, :3] - grav); X[b, :, 3:] = sp * imu[:, 3:]
            v = sp * vel
            Yw[b] = v.reshape(T, WIN, 3).mean(1); Yd[b] = v.reshape(T * TOK, WIN // TOK, 3).mean(1)
        elif n >= T:
            s = rng.integers(0, n - T + 1)
            X[b] = t["imu"][s * WIN:(s + T) * WIN]; Yw[b] = t["win_v"][s:s + T]; Yd[b] = t["dense_v"][s * TOK:(s + T) * TOK]
        else:
            X[b, : n * WIN] = t["imu"]; X[b, n * WIN:] = t["imu"][-1]
            Yw[b, :n] = t["win_v"]; Yw[b, n:] = 0; Yd[b, : n * TOK] = t["dense_v"]; Yd[b, n * TOK:] = 0; M[b, n:] = 0
        P[b] = t["plat"]; LG[b] = t["lag"]
    return X, Yw, Yd, M, P, LG


def rand_rotation(B, max_deg):
    """Small random rotations (B,3,3): uniform random axis, angle ~ U(0, max_deg)."""
    axis = torch.randn(B, 3, device=device); axis = axis / axis.norm(dim=1, keepdim=True)
    ang = torch.rand(B, 1, device=device) * math.radians(max_deg)
    K = torch.zeros(B, 3, 3, device=device)
    K[:, 0, 1], K[:, 0, 2], K[:, 1, 0] = -axis[:, 2], axis[:, 1], axis[:, 2]
    K[:, 1, 2], K[:, 2, 0], K[:, 2, 1] = -axis[:, 0], -axis[:, 1], axis[:, 0]
    I = torch.eye(3, device=device).expand(B, 3, 3)
    s, c = ang.sin().view(B, 1, 1), ang.cos().view(B, 1, 1)
    return I + s * K + (1 - c) * (K @ K)


def augment(X, Yw, Yd, P):
    """Physically consistent augmentation on device. X:(B,L,6) Yw:(B,T,3) Yd:(B,N,3) P:(B,) platform ids."""
    B = X.shape[0]
    R = rand_rotation(B, args.rot_deg)                                   # re-mount the sensor
    if args.rot_deg_drone > 0:
        Rd = rand_rotation(B, args.rot_deg_drone)
        R = torch.where((P == PLAT2ID["drone"]).view(B, 1, 1), Rd, R)
    acc, gyr = X[..., :3] @ R.transpose(1, 2), X[..., 3:] @ R.transpose(1, 2)
    Yw, Yd = Yw @ R.transpose(1, 2), Yd @ R.transpose(1, 2)
    acc = acc * (1 + 0.02 * torch.randn(B, 1, 3, device=device)) + 0.15 * torch.randn(B, 1, 3, device=device)
    gyr = gyr * (1 + 0.02 * torch.randn(B, 1, 3, device=device)) + 0.02 * torch.randn(B, 1, 3, device=device)
    acc = acc + 0.05 * torch.randn_like(acc); gyr = gyr + 0.004 * torch.randn_like(gyr)
    X = torch.cat([acc, gyr], -1)
    if args.vib > 1:                                                     # randomise vibration amplitude (high-frequency part)
        lp = F.avg_pool1d(X.transpose(1, 2), 9, stride=1, padding=4, count_include_pad=False).transpose(1, 2)
        s = torch.exp(torch.empty(B, 1, 2, device=device).uniform_(-math.log(args.vib), math.log(args.vib)))
        s = torch.cat([s[..., :1].expand(B, 1, 3), s[..., 1:].expand(B, 1, 3)], -1)  # one factor for accel, one for gyro
        X = lp + s * (X - lp)
    return X, Yw, Yd


def vhuber(pred, tgt, beta=0.25, mask=None):
    e = torch.linalg.vector_norm(pred - tgt, dim=-1)
    l = torch.where(e < beta, 0.5 * e.square() / beta, e - 0.5 * beta)
    return l.mean() if mask is None else (l * mask).sum() / mask.sum().clamp(min=1)


# --------------------------------------------------------------------------- validation
val_sol, val_trajs = None, None
if "val" not in args.splits.split(","):
    val_sol = build_solution("val")
    val_trajs = load_split("val", keys=("imu",))
    val_idx = read_index("val")


def evaluate(m):
    m.eval()
    preds = []
    for tid, g in val_idx.groupby("traj_id", sort=False):
        v = predict_trajectory(m, val_trajs[tid]["imu"], device, chunk_win=T)
        preds.append(pd.DataFrame({"window_id": g["window_id"].to_numpy(), "vx": v[g.win_idx, 0], "vy": v[g.win_idx, 1], "vz": v[g.win_idx, 2]}))
    m.train()
    return score_predictions(val_sol, pd.concat(preds, ignore_index=True))


# --------------------------------------------------------------------------- train
model = IMUNet(width=args.width, ctx_layers=args.ctx_layers, physics=bool(args.physics), lag=bool(args.lag)).to(device)
ema = deepcopy(model).eval()
for q in ema.parameters():
    q.requires_grad_(False)
print(f"{sum(q.numel() for q in model.parameters())/1e6:.2f}M params")
opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd, betas=(0.9, 0.99))
total_steps = args.epochs * args.steps
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=args.lr, total_steps=total_steps, pct_start=0.08, div_factor=20, final_div_factor=200)
ema_decay = 0.998
log, best = [], float("inf")
swa_state, swa_n = None, 0
t0 = time.time()
for ep in range(1, args.epochs + 1):
    model.train(); tot = {"win": 0., "dense": 0., "drift": 0., "plat": 0., "lag": 0.}
    for it in range(args.steps):
        X, Yw, Yd, M, P, LG = sample_batch(args.batch)
        X, Yw, Yd = (torch.from_numpy(a).to(device) for a in (X, Yw, Yd))
        M, P, LG = torch.from_numpy(M).to(device), torch.from_numpy(P).to(device), torch.from_numpy(LG).to(device)
        X, Yw, Yd = augment(X, Yw, Yd, P)
        dense, plat, delta = model(X.transpose(1, 2), return_lag=True)
        l_lag = F.smooth_l1_loss(delta, LG, beta=0.2) if delta is not None else torch.zeros((), device=device)
        pw = model.to_windows(dense)
        l_win = vhuber(pw, Yw, mask=M)
        l_dense = vhuber(dense, Yd, mask=M.repeat_interleave(TOK, dim=1))
        cum = torch.cumsum((pw - Yw) * M[..., None], dim=1)                    # integrated body-frame error
        l_drift = (torch.linalg.vector_norm(cum, dim=-1) / torch.sqrt(torch.arange(1, T + 1, device=device))).mean()
        l_plat = F.cross_entropy(plat, P)
        loss = l_win + 0.5 * l_dense + 0.2 * l_drift + 0.05 * l_plat + 0.5 * l_lag
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        opt.step(); sched.step()
        with torch.no_grad():
            d = min(ema_decay, (1 + ep * args.steps + it) / (10 + ep * args.steps + it))
            for pe, pm in zip(ema.parameters(), model.parameters()):
                pe.mul_(d).add_(pm.detach(), alpha=1 - d)
        for k, v in zip(tot, (l_win, l_dense, l_drift, l_plat, l_lag)):
            tot[k] += v.item() / args.steps
    rec = {"epoch": ep, "lr": sched.get_last_lr()[0], "min": (time.time() - t0) / 60, **{f"l_{k}": round(v, 4) for k, v in tot.items()}}
    if val_sol is not None and (ep % args.eval_every == 0 or ep == args.epochs):
        s, pp = evaluate(ema)
        rec["val_score"] = round(s, 4); rec.update({f"{k}_score": round(v, 3) for k, v in pp["score"].items()})
        rec.update({f"{k}_ave": round(v, 3) for k, v in pp["ave"].items()}); rec.update({f"{k}_ate": round(v, 3) for k, v in pp["ate20"].items()})
        if s < best:
            best = s; torch.save({"model": ema.state_dict(), "args": vars(args), "val_score": s, "epoch": ep}, run / "best.pt")
    torch.save({"model": ema.state_dict(), "args": vars(args), "epoch": ep}, run / "last.pt")
    if args.swa_from and ep >= args.swa_from:                                  # running uniform average of EMA weights
        sd = {k: v.detach().clone().float() for k, v in ema.state_dict().items()}
        swa_state = sd if swa_state is None else {k: swa_state[k] + (sd[k] - swa_state[k]) / (swa_n + 1) for k in sd}
        swa_n += 1
        torch.save({"model": swa_state, "args": vars(args), "epoch": ep, "swa_n": swa_n}, run / "swa.pt")
    log.append(rec); pd.DataFrame(log).to_csv(run / "log.csv", index=False)
    print(json.dumps(rec))
print("best val score", best)


In [ ]:
%%writefile predict.py
"""Generate a Kaggle submission from one or more checkpoints (checkpoint ensembling = mean of velocities).

  python predict.py --ckpt runs/v1/best.pt [runs/v2/best.pt ...] --out submission.csv [--val]

--val also scores the checkpoint(s) on the val split with the official metric before predicting test.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from common import DATA, read_index, load_split, build_solution, score_predictions
from model import IMUNet, predict_trajectory

p = argparse.ArgumentParser()
p.add_argument("--ckpt", nargs="+", required=True)
p.add_argument("--out", default="submission.csv")
p.add_argument("--stride", type=int, default=2, help="chunk stride in windows (smaller = more overlap averaging)")
p.add_argument("--val", action="store_true")
p.add_argument("--tta-deg", type=float, default=0.0, help="rotation TTA: ±deg about each body axis (0 = off)")
args = p.parse_args()
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

models = []
for c in args.ckpt:
    ck = torch.load(c, map_location="cpu")
    m = IMUNet(width=ck["args"].get("width", 128), ctx_layers=ck["args"].get("ctx_layers", 2), physics=bool(ck["args"].get("physics", 0)), lag=bool(ck["args"].get("lag", 0))).to(device).eval()
    m.load_state_dict(ck["model"]); models.append((m, ck["args"].get("chunk", 16)))
    print(f"loaded {c}: epoch {ck.get('epoch')}, val_score {ck.get('val_score', 'n/a')}")


def tta_rotations(deg):
    """Identity plus ±deg rotations about each body axis (sensor re-mount TTA)."""
    from scipy.spatial.transform import Rotation
    Rs = [np.eye(3)]
    if deg > 0:
        for ax in "xyz":
            for sgn in (1, -1):
                Rs.append(Rotation.from_euler(ax, sgn * deg, degrees=True).as_matrix())
    return [R.astype(np.float32) for R in Rs]


def predict_traj_tta(m, imu, cw):
    vs = []
    for R in tta_rotations(args.tta_deg):
        x = np.concatenate([imu[:, :3] @ R.T, imu[:, 3:] @ R.T], axis=1)       # rotate sensor frame
        vs.append(predict_trajectory(m, x, device, chunk_win=cw, stride_win=args.stride) @ R)  # rotate velocity back
    return np.mean(vs, axis=0)


def predict_split(split):
    idx = read_index(split); trajs = load_split(split, keys=("imu",)); out = []
    for tid, g in idx.groupby("traj_id", sort=False):
        v = np.mean([predict_traj_tta(m, trajs[tid]["imu"], cw) for m, cw in models], axis=0)
        w = g["win_idx"].to_numpy()
        out.append(pd.DataFrame({"window_id": g["window_id"].to_numpy(), "vx": v[w, 0], "vy": v[w, 1], "vz": v[w, 2]}))
    return pd.concat(out, ignore_index=True)


if args.val:
    s, pp = score_predictions(build_solution("val"), predict_split("val"))
    print(f"val TartanIMU score: {s:.4f}\n{pp.round(3)}")

sub = pd.read_csv(DATA / "sample_submission.csv")[["window_id"]].merge(predict_split("test"), on="window_id", how="left")
assert len(sub) == 30644 and not sub.isna().any().any(), "submission incomplete"
sub.to_csv(args.out, index=False)
speed = np.linalg.norm(sub[["vx", "vy", "vz"]].to_numpy(), axis=1)
print(f"wrote {args.out}: {len(sub)} rows; speed median {np.median(speed):.3f}, p99 {np.quantile(speed, .99):.3f}, max {speed.max():.3f}")


In [ ]:
%%writefile breakdown.py
"""Val breakdown by platform and drone source (A = racing drone, B = the rest) for one or more checkpoints.
  python breakdown.py runs/v1/best.pt [runs/v3/best.pt ...]
"""
import sys, numpy as np, pandas as pd, torch
from common import load_split, read_index, build_solution
from kaggle_metric import _ate_traj, _ave_traj, AVE_REF, ATE_REF
from model import IMUNet, predict_trajectory
pd.set_option("display.width", 250)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
sol = build_solution("val"); idx = read_index("val"); trajs = load_split("val", keys=("imu",))
for ckpt in sys.argv[1:]:
    ck = torch.load(ckpt, map_location="cpu", weights_only=False); a = ck["args"]
    m = IMUNet(width=a.get("width", 128), ctx_layers=a.get("ctx_layers", 2), physics=bool(a.get("physics", 0)), lag=bool(a.get("lag", 0))).to(device).eval(); m.load_state_dict(ck["model"])
    preds = []
    for tid, g in idx.groupby("traj_id", sort=False):
        v = predict_trajectory(m, trajs[tid]["imu"], device, chunk_win=a.get("chunk", 16), stride_win=2)
        w = g["win_idx"].to_numpy(); preds.append(pd.DataFrame({"window_id": g["window_id"].to_numpy(), "vx_pred": v[w, 0], "vy_pred": v[w, 1], "vz_pred": v[w, 2]}))
    mm = sol.merge(pd.concat(preds), on="window_id")
    recs = []
    for tid, g in mm.groupby("traj_id", sort=False):
        g = g.sort_values("win_idx"); p = g["platform"].iloc[0]
        grp = p if p != "drone" else ("drone-A" if int(tid[-4:]) <= 8 else "drone-B")
        recs.append({"platform": p, "group": grp, "ave": _ave_traj(g), "ate20": _ate_traj(g)})
    r = pd.DataFrame(recs)
    pp = r.groupby("platform")[["ave", "ate20"]].mean(); score = 0.6 * pp["ave"].mean() / AVE_REF + 0.4 * pp["ate20"].mean() / ATE_REF
    gg = r.groupby("group")[["ave", "ate20"]].mean(); gg["n_traj"] = r.groupby("group").size()
    print(f"\n== {ckpt} (epoch {ck.get('epoch')}): val score {score:.4f}")
    print(gg.round(3).to_string())


## 3. Validated experiments (train on `train`, score on `val`)

Each run: 60 epochs × 250 steps, EMA + SWA over the last 10 epochs, val every 4 epochs. `breakdown.py` prints AVE / ATE20
per platform and separately for the two drone sources (**drone-A** = racing drone, the hardest and the one that dominates
the test set; **drone-B** = the rest). Compare against the local reference lines in the table below.

| run | change | local val (M5) |
| --- | --- | --- |
| E0 | `v4` recipe, width 128 — reference (its 160-epoch train+val version is the current LB best 0.2870) | 0.2030 |
| E1 | E0 without drone source B in training (does B's odd sensor convention hurt A?) | ? |
| E3 | E0 + vibration-amplitude augmentation (×0.33–3) | ? |
| E5 | E0 with uniform per-trajectory sampling (the metric weights trajectories equally, not by length) | ? |
| E6 | E0 + 24 s context chunks | ? |

Each 60-epoch run takes ~45 min on an L4 (~15 min on an A100). Edit the dict to add/remove experiments;
finished runs are skipped on re-execution, so the cell can be re-run after a disconnect.

In [ ]:
import subprocess, json, shlex
COMMON = "--epochs 60 --steps 250 --eval-every 4 --physics 1 --dilate 1.3 --rot-deg-drone 45 --swa-from 50"
EXPERIMENTS = {
    "E0": "--boost-a 4",
    "E1": "--boost-a 4 --drop-source-b 1",
    "E3": "--boost-a 4 --vib 3",
    "E5": "--boost-a 1 --traj-uniform 1",
    "E6": "--boost-a 4 --chunk 24",
}
KEEP = ("epoch", "val_score", "car_score", "dog_score", "drone_score", "human_score", "drone_ave")


def run_train(name, flags):
    """Run train.py with live, filtered output; raise (with the tail of the log) if it fails."""
    cmd = ["python", "train.py", "--name", name] + shlex.split(flags)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line.rstrip()])[-25:]
        if line.startswith('{"epoch"'):
            rec = json.loads(line)
            if "val_score" in rec:
                print({k: rec[k] for k in KEEP if k in rec}, flush=True)
    p.wait()
    if p.returncode != 0:
        print("\n".join(tail)); raise RuntimeError(f"{name} failed (exit {p.returncode}) — see log tail above")


for name, extra in EXPERIMENTS.items():
    if pathlib.Path(f"runs/{name}/swa.pt").exists():
        print(f"{name}: already done, skipping"); continue
    t0 = time.time(); print(f"\n======== {name}: {extra}", flush=True)
    run_train(name, f"{COMMON} {extra}")
    print(f"{name} done in {(time.time()-t0)/60:.1f} min")

In [ ]:
# per-platform / per-drone-source breakdown of every finished experiment (lower is better everywhere)
import glob
ckpts = sorted(glob.glob("runs/E*/swa.pt"))
out = subprocess.run(["python", "breakdown.py"] + ckpts, capture_output=True, text=True)
print("\n".join(l for l in (out.stdout + out.stderr).splitlines() if "Warning" not in l))

In [ ]:
# overfitting check: train loss vs val score curves
import pandas as pd, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for name in EXPERIMENTS:
    p = pathlib.Path(f"runs/{name}/log.csv")
    if not p.exists(): continue
    d = pd.read_csv(p); ax[0].plot(d.epoch, d.l_win, label=name); dv = d[d.val_score.notna()]; ax[1].plot(dv.epoch, dv.val_score, "o-", label=name)
ax[0].set_title("train window loss"); ax[1].set_title("val TartanIMU score (disjoint trajectories)"); ax[1].set_ylim(0.18, 0.30)
for a in ax: a.legend(); a.grid(alpha=.3)
plt.show()

## 4. Final model: best recipe on train+val, fixed long schedule → submission

Set `BEST` to the experiment flags that won on val. The schedule length is the one lever that kept paying on the
leaderboard (100 → 160 epochs: 0.2975 → 0.2870), so the final uses 240 epochs with SWA over the last 30.
No selection is done on the leaderboard: this is one run, one checkpoint.

In [ ]:
BEST = EXPERIMENTS["E0"]           # <- replace with the winning flags from section 3
FINAL = "final_colab2"                # new name each time, so an old run is not skipped
t0 = time.time()
if not pathlib.Path(f"runs/{FINAL}/swa.pt").exists():
    run_train(FINAL, f"--splits train,val --epochs 160 --steps 250 --physics 1 --dilate 1.3 --rot-deg-drone 45 --swa-from 140 {BEST}")
print(f"final training: {(time.time()-t0)/60:.1f} min")
out = subprocess.run(["python", "predict.py", "--ckpt", f"runs/{FINAL}/swa.pt", "--out", f"submission_{FINAL}.csv"], capture_output=True, text=True)
print("\n".join(l for l in (out.stdout + out.stderr).splitlines() if "Warning" not in l))

In [ ]:
# submit (uses 1 of the 5 daily submissions) and record the id
!kaggle competitions submit -c tartan-imu-challenge-iros2026 -f submission_{FINAL}.csv -m "{FINAL}: {BEST}, train+val, 160 ep, EMA+SWA — single model"
time.sleep(90)
!kaggle competitions submissions tartan-imu-challenge-iros2026 | head -4

In [ ]:
# keep the artefacts: checkpoint (+ its SHA-256 for the report), config, submission -> Google Drive
import hashlib, shutil
print("checkpoint SHA-256:", hashlib.sha256(open(f"runs/{FINAL}/swa.pt","rb").read()).hexdigest())
from google.colab import drive; drive.mount("/content/drive")
out = pathlib.Path("/content/drive/MyDrive/tartanimu"); out.mkdir(exist_ok=True)
for f in [f"runs/{FINAL}/swa.pt", f"runs/{FINAL}/log.csv", f"submission_{FINAL}.csv"]:
    shutil.copy(f, out / pathlib.Path(f).name.replace("swa.pt", f"{FINAL}_swa.pt"))
print("saved to", out, list(p.name for p in out.iterdir()))